In [1]:
import os
import sys

print("=" * 70)
print("Day 40 收官项目：Banking Operations Agent 自动化集成测试")
print("=" * 70)

thread_config = {"configurable": {"thread_id": "day40-prod-thread-001"}}

from agent_graph import build_banking_graph
from banking_backend import ProductionBankingBackend
from langgraph.types import Command
import sqlite3

# 辅助函数：清空数据库表，避免 Windows 文件删除锁死问题
def reset_databases():
    for db in ["official_checkpoints.db", "banking_ledger.db"]:
        if os.path.exists(db):
            try:
                os.remove(db)
            except PermissionError:
                # 如果文件被占用，则连接并清空表数据，替代暴力删除文件
                with sqlite3.connect(db) as conn:
                    cursor = conn.cursor()
                    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
                    for table in cursor.fetchall():
                        if table[0] != 'sqlite_sequence':
                            conn.execute(f"DELETE FROM {table[0]};")
                    conn.commit()

# 初始化清理
reset_databases()

# 场景 1：纯余额查询（自动执行，无审批）
print("\n--- 场景 1: 查询余额 ---")
graph, conn = build_banking_graph()
res1 = graph.invoke({"messages": [{"role": "user", "content": "帮我查一下账户余额"}]}, config=thread_config)
print("余额查询执行完成，消息记录数:", len(res1["messages"]))
conn.close()

# 重置数据库以进入场景 2
reset_databases()

print("\n--- 场景 2: 发起转账请求（应在审批节点中断挂起） ---")
graph_a, conn_a = build_banking_graph()
graph_a.invoke({"messages": [{"role": "user", "content": "请帮我查询余额并转账 1000 元给 Alice"}]}, config=thread_config)

snapshot_a = graph_a.get_state(thread_config)
print(f"进程 A 挂起节点: {snapshot_a.next}")
assert "approval_node" in snapshot_a.next
conn_a.close()
print("进程 A 成功挂起并落盘")

print("\n--- 场景 3: 模拟进程重启，加载 DB 并通过 Command(resume=...) 批准转账 ---")
graph_b, conn_b = build_banking_graph()
snapshot_b = graph_b.get_state(thread_config)
print(f"进程 B 成功恢复节点: {snapshot_b.next}")

resume_cmd = Command(resume={
    "approved": True,
    "tool_call_id": "call_trans_01",
    "args": {"amount": 1000.0, "recipient": "Alice", "idempotency_key": "idempotency-txn-day40-01"}
})

graph_b.invoke(resume_cmd, config=thread_config)
conn_b.close()

backend = ProductionBankingBackend()
print(f"转账完成后的最终账户余额: {backend.get_balance()}")
assert backend.get_balance() == 4000.0
print("Day 40 收官集成测试全部顺利通过！")

Day 40 收官项目：Banking Operations Agent 自动化集成测试

--- 场景 1: 查询余额 ---
余额查询执行完成，消息记录数: 3

--- 场景 2: 发起转账请求（应在审批节点中断挂起） ---
进程 A 挂起节点: ('approval_node',)
进程 A 成功挂起并落盘

--- 场景 3: 模拟进程重启，加载 DB 并通过 Command(resume=...) 批准转账 ---
进程 B 成功恢复节点: ('approval_node',)
转账完成后的最终账户余额: 4000.0
Day 40 收官集成测试全部顺利通过！
